# 🏆 Datathon Final Pipeline
## Bilişsel Performans Skoru Tahmini — RMSE Regresyon

**Hedef:** `bilissel_performans_skoru`  
**Metrik:** RMSE  
**En İyi Public RMSE:** `1.20244`  
**Final Yaklaşım:** CatBoost ağırlıklı OOF blend (HGB + LightGBM + XGBoost + CatBoost)

---
### Pipeline Özeti
| Aşama | Açıklama |
|---|---|
| **Stage 4** | HGB + Target Encoding hiperparametre taraması (9 config) |
| **Stage 6** | LightGBM / XGBoost / CatBoost modelleri (8 config) |
| **Stage 7** | CatBoost feature set ablasyonu → top-3 blend |
| **Stage 8** | CatBoost fine-tuning + 5 seed averaging |
| **Stage 9** | OOF tabanlı seçilmiş blend optimizasyonu |

> **Not:** Veriyi Kaggle'dan indirip `data/` klasörüne koyun:  
> `data/train.csv`, `data/test_x.csv`, `data/sample_submission.csv`


## 0. Kurulum ve Klasör Yapısı

In [ ]:
# Gerekli paketleri yükle (ilk çalıştırmada)
# !pip install pandas numpy scikit-learn lightgbm xgboost catboost

import os, shutil
from pathlib import Path

# Proje kök dizini bu notebook'un bulunduğu yerin bir üstü olmalı
# Notebook notebooks/ altındaysa:
PROJECT_ROOT = Path.cwd().parent
# Eğer notebook proje kökünde çalışıyorsa:
# PROJECT_ROOT = Path.cwd()

for d in ["data", "submissions", "reports/experiments", "src"]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

print("Proje kök dizini:", PROJECT_ROOT)
print("Klasörler hazır.")


In [ ]:
# sys.path ayarı — src/ modülleri import edilebilsin
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Temel kütüphaneler
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print("Python sürümü:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


---
## 1. Experiment Framework (`experiment_framework.py`)

Tüm pipeline'ın temel modülü. İçerir:
- Sabitler: `TARGET`, `ID_COL`, `RANDOM_STATE`
- `TargetMeanEncoder`: CV-safe smoothed target encoding
- Feature engineering fonksiyonları (ülke normalizasyonu, uyku özellikleri, interaksiyonlar, binler, forensik kombinasyonlar)
- `run_cv_experiment`: Stratified 5-fold CV ile model eğitim/kaydetme döngüsü
- `blend_experiments`: OOF tabanlı blend üretimi


In [ ]:
# ─── experiment_framework.py ─────────────────────────────────────────────────
from __future__ import annotations

import importlib.util
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


TARGET = "bilissel_performans_skoru"
ID_COL = "id"
RANDOM_STATE = 42


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def package_exists(package_name: str) -> bool:
    return importlib.util.find_spec(package_name) is not None


def project_root() -> Path:
    return PROJECT_ROOT


def load_data():
    root = project_root()
    train = pd.read_csv(root / "data" / "train.csv")
    test  = pd.read_csv(root / "data" / "test_x.csv")
    return train, test


print("Framework sabitleri yüklendi.")
print(f"  TARGET        = {TARGET}")
print(f"  ID_COL        = {ID_COL}")
print(f"  RANDOM_STATE  = {RANDOM_STATE}")


In [ ]:
# ─── Target Mean Encoder ──────────────────────────────────────────────────────
class TargetMeanEncoder(BaseEstimator, TransformerMixin):
    """Cross-validation-safe smoothed target mean encoder."""

    def __init__(self, smoothing: float = 25.0):
        self.smoothing = smoothing

    def fit(self, X, y):
        X = pd.DataFrame(X).copy().reset_index(drop=True)
        y = pd.Series(y).reset_index(drop=True)
        self.columns_ = X.columns.tolist()
        self.global_mean_ = float(y.mean())
        self.maps_ = {}
        for col in self.columns_:
            values = X[col].astype("object").where(X[col].notna(), "__MISSING__")
            stats = (
                pd.DataFrame({"value": values, "target": y})
                .groupby("value")["target"]
                .agg(["mean", "count"])
            )
            smooth_mean = (
                stats["mean"] * stats["count"] + self.global_mean_ * self.smoothing
            ) / (stats["count"] + self.smoothing)
            self.maps_[col] = smooth_mean.to_dict()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        if len(X.columns) == len(self.columns_):
            X.columns = self.columns_
        encoded_cols = []
        for col in self.columns_:
            values = X[col].astype("object").where(X[col].notna(), "__MISSING__")
            encoded = values.map(self.maps_[col]).fillna(self.global_mean_).astype(float)
            encoded_cols.append(encoded.to_numpy())
        return np.column_stack(encoded_cols)

print("TargetMeanEncoder tanımlandı.")


In [ ]:
# ─── Feature Engineering Fonksiyonları ───────────────────────────────────────

def normalize_country(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ulke_map = {"South Korea": "Guney Kore", "Spain": "Ispanya", "Sweden": "Isvec"}
    df["ulke"] = df["ulke"].replace(ulke_map)
    return df


def add_missing_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    missing_cols = ["meslek", "vucut_kitle_indeksi", "uyku_oncesi_kafein_mg",
                    "stres_skoru", "kronotip", "ruh_sagligi_durumu"]
    for col in missing_cols:
        df[f"{col}_missing"] = df[col].isna().astype(int)
    return df


def add_sleep_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["toplam_kaliteli_uyku_yuzdesi"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]
    df["hafif_uyku_yuzdesi"] = 100 - df["rem_yuzdesi"] - df["derin_uyku_yuzdesi"]
    df["uyku_kalite_orani"] = df["toplam_kaliteli_uyku_yuzdesi"] / (df["hafif_uyku_yuzdesi"].clip(lower=1) + 1)
    df["rem_derin_orani"]  = df["rem_yuzdesi"] / (df["derin_uyku_yuzdesi"] + 1)
    df["derin_rem_orani"]  = df["derin_uyku_yuzdesi"] / (df["rem_yuzdesi"] + 1)
    return df


def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["kaliteli_uyku_stres_orani"]  = (df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]) / (df["stres_skoru"] + 1)
    df["stres_calisma_etkilesimi"]   = df["stres_skoru"] * df["gunluk_calisma_saati"]
    df["stres_uyanma_etkilesimi"]    = df["stres_skoru"] * df["gecelik_uyanma_sayisi"]
    df["stres_gecikme_etkilesimi"]   = df["stres_skoru"] * df["uykuya_dalma_suresi_dk"]
    df["uyku_bolunme_yuku"]          = df["gecelik_uyanma_sayisi"] * df["uykuya_dalma_suresi_dk"]
    df["ekran_kafein_etkilesimi"]    = df["uyku_oncesi_ekran_suresi_dk"] * df["uyku_oncesi_kafein_mg"]
    df["aktivite_stres_orani"]       = df["gunluk_adim_sayisi"] / (df["stres_skoru"] + 1)
    df["adim_calisma_orani"]         = df["gunluk_adim_sayisi"] / (df["gunluk_calisma_saati"] + 1)
    df["uyku_hijyen_riski"] = (
        df["uyku_oncesi_ekran_suresi_dk"] / 60
        + df["uyku_oncesi_kafein_mg"] / 100
        + df["uykuya_dalma_suresi_dk"] / 30
        + df["gecelik_uyanma_sayisi"]
    )
    return df


def add_health_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["hafta_sonu_mu"]    = (df["gun_tipi"] == "Hafta sonu").astype(int)
    df["saglikli_mi"]      = (df["ruh_sagligi_durumu"] == "Saglikli").astype(int)
    df["depresyon_var_mi"] = df["ruh_sagligi_durumu"].isin(["Depresyon", "Anksiyete ve depresyon"]).astype(int)
    df["anksiyete_var_mi"] = df["ruh_sagligi_durumu"].isin(["Anksiyete", "Anksiyete ve depresyon"]).astype(int)
    df["sabah_insani_mi"]  = (df["kronotip"] == "Sabah insani").astype(int)
    df["gece_insani_mi"]   = (df["kronotip"] == "Gece insani").astype(int)
    return df


def add_binned_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["yas_grup"]     = pd.cut(df["yas"], bins=[17,24,34,44,54,69,120],
                                labels=["18-24","25-34","35-44","45-54","55-69","70+"]).astype("object")
    df["bmi_grup"]     = pd.cut(df["vucut_kitle_indeksi"], bins=[0,18.5,25,30,35,100],
                                labels=["zayif","normal","fazla_kilo","obez_1","obez_2"]).astype("object")
    df["stres_grup"]   = pd.cut(df["stres_skoru"], bins=[0,3,6,8,10],
                                labels=["dusuk","orta","yuksek","cok_yuksek"], include_lowest=True).astype("object")
    df["calisma_grup"] = pd.cut(df["gunluk_calisma_saati"], bins=[-1,4,8,12,24],
                                labels=["az","normal","uzun","cok_uzun"]).astype("object")
    df["ekran_grup"]   = pd.cut(df["uyku_oncesi_ekran_suresi_dk"], bins=[0,30,60,120,240],
                                labels=["dusuk","orta","yuksek","cok_yuksek"], include_lowest=True).astype("object")
    df["adim_grup"]    = pd.cut(df["gunluk_adim_sayisi"], bins=[0,3000,7000,10000,30000],
                                labels=["dusuk","orta","iyi","yuksek"], include_lowest=True).astype("object")
    return df


def _combo_value(df: pd.DataFrame, cols: list) -> pd.Series:
    values = [df[col].astype("object").where(df[col].notna(), "__MISSING__").astype(str) for col in cols]
    return values[0].str.cat(values[1:], sep="__")


def add_forensics_group_interactions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    combos = [
        ["meslek","ruh_sagligi_durumu"],
        ["meslek","ruh_sagligi_durumu","gun_tipi"],
        ["meslek","ruh_sagligi_durumu","mevsim"],
        ["cinsiyet","meslek","ruh_sagligi_durumu"],
        ["meslek","kronotip","ruh_sagligi_durumu"],
        ["meslek","ulke","ruh_sagligi_durumu"],
        ["kronotip","ruh_sagligi_durumu","gun_tipi"],
        ["ruh_sagligi_durumu","gun_tipi"],
        ["ruh_sagligi_durumu","mevsim","gun_tipi"],
        ["meslek","gun_tipi"],
        ["meslek","mevsim","gun_tipi"],
        ["meslek","kronotip","gun_tipi"],
        ["ulke","ruh_sagligi_durumu","gun_tipi"],
    ]
    for cols in combos:
        df["combo__" + "__".join(cols)] = _combo_value(df, cols)
    return df


def add_forensics_risk_bins(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["stres_detay_grup"] = pd.cut(df["stres_skoru"],
        bins=[-0.01,3.0,5.0,6.2,6.62,7.1,7.74,10.0],
        labels=["s_0_3","s_3_5","s_5_62","s_62_66","s_66_71","s_71_77","s_77_10"],
        include_lowest=True).astype("object")
    df["calisma_detay_grup"] = pd.cut(df["gunluk_calisma_saati"],
        bins=[-0.01,4.0,7.36,8.28,9.21,10.22,11.52,24.0],
        labels=["c_0_4","c_4_736","c_736_828","c_828_921","c_921_1022","c_1022_1152","c_1152_plus"],
        include_lowest=True).astype("object")
    df["rem_detay_grup"] = pd.cut(df["rem_yuzdesi"],
        bins=[-0.01,15.78,17.36,18.48,19.44,20.30,23.0,100.0],
        labels=["r_low","r_158_174","r_174_185","r_185_194","r_194_203","r_203_23","r_high"],
        include_lowest=True).astype("object")
    df["uykuya_dalma_detay_grup"] = pd.cut(df["uykuya_dalma_suresi_dk"],
        bins=[-0.01,10,15,20,24,27,31,60,240],
        labels=["lat_0_10","lat_10_15","lat_15_20","lat_20_24","lat_24_27","lat_27_31","lat_31_60","lat_60_plus"],
        include_lowest=True).astype("object")
    df["uyanma_detay_grup"] = pd.cut(df["gecelik_uyanma_sayisi"],
        bins=[-0.01,1,2,3,4,5,6,8,20],
        labels=["w_0_1","w_1_2","w_2_3","w_3_4","w_4_5","w_5_6","w_6_8","w_8_plus"],
        include_lowest=True).astype("object")
    df["kafein_detay_grup"] = pd.cut(df["uyku_oncesi_kafein_mg"],
        bins=[-0.01,0,25,75,150,400,1000],
        labels=["caf_0","caf_1_25","caf_25_75","caf_75_150","caf_150_400","caf_400_plus"],
        include_lowest=True).astype("object")
    return df


def add_forensics_risk_interactions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    combos = [
        ["ruh_sagligi_durumu","stres_detay_grup"],
        ["meslek","stres_detay_grup"],
        ["meslek","ruh_sagligi_durumu","stres_detay_grup"],
        ["meslek","calisma_detay_grup"],
        ["gun_tipi","calisma_detay_grup"],
        ["ruh_sagligi_durumu","calisma_detay_grup"],
        ["stres_detay_grup","calisma_detay_grup"],
        ["stres_detay_grup","uyanma_detay_grup"],
        ["ruh_sagligi_durumu","uyanma_detay_grup"],
        ["rem_detay_grup","stres_detay_grup"],
        ["uykuya_dalma_detay_grup","stres_detay_grup"],
        ["kafein_detay_grup","stres_detay_grup"],
    ]
    for cols in combos:
        df["combo__" + "__".join(cols)] = _combo_value(df, cols)
    return df


FEATURE_SET_STEPS = {
    "fs_00_raw":                  [],
    "fs_01_clean_country":        [normalize_country],
    "fs_02_missing_indicators":   [normalize_country, add_missing_indicators],
    "fs_03_sleep_features":       [normalize_country, add_sleep_features],
    "fs_04_interactions":         [normalize_country, add_sleep_features, add_interaction_features],
    "fs_05_health_flags":         [normalize_country, add_health_flags],
    "fs_06_bins":                 [normalize_country, add_binned_features],
    "fs_07_all_safe":             [normalize_country, add_missing_indicators, add_sleep_features,
                                   add_interaction_features, add_health_flags, add_binned_features],
    "fs_08_forensic_combos":      [normalize_country, add_missing_indicators, add_forensics_group_interactions],
    "fs_09_forensic_risk_bins":   [normalize_country, add_missing_indicators, add_forensics_risk_bins],
    "fs_10_forensic_combo_risk":  [normalize_country, add_missing_indicators, add_forensics_group_interactions,
                                   add_forensics_risk_bins, add_forensics_risk_interactions],
}


def apply_feature_set(df: pd.DataFrame, feature_set: str) -> pd.DataFrame:
    if feature_set not in FEATURE_SET_STEPS:
        raise ValueError(f"Bilinmeyen feature_set={feature_set}. Geçerliler: {', '.join(FEATURE_SET_STEPS)}")
    out = df.copy()
    for step in FEATURE_SET_STEPS[feature_set]:
        out = step(out)
    return out


def available_feature_sets() -> list:
    return list(FEATURE_SET_STEPS.keys())


print(f"Feature engineering fonksiyonları tanımlandı.")
print(f"Toplam feature set sayısı: {len(FEATURE_SET_STEPS)}")


In [ ]:
# ─── Preprocessor & Model Config ─────────────────────────────────────────────

def make_onehot_encoder(dense: bool):
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=not dense)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=not dense)


def make_preprocessor(numeric_features, categorical_features, encoding, dense,
                       smoothing=25.0, scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    if encoding == "onehot":
        categorical_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
            ("one_hot", make_onehot_encoder(dense=dense)),
        ])
    elif encoding == "target":
        categorical_pipeline = TargetMeanEncoder(smoothing=smoothing)
    else:
        raise ValueError("encoding 'onehot' veya 'target' olmalı")

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=numeric_steps), numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ],
        sparse_threshold=0.0 if dense else 0.3,
    )


@dataclass(frozen=True)
class ModelConfig:
    name: str
    encoding: str
    dense: bool
    estimator: object
    smoothing: float = 25.0
    scale_numeric: bool = False


def build_pipeline(model_name, numeric_features, categorical_features):
    """Sklearn model config registry'den pipeline üretir."""
    configs = {
        "hgb_te_base": ModelConfig(
            name="hgb_te_base", encoding="target", dense=True, smoothing=25.0,
            estimator=HistGradientBoostingRegressor(
                max_iter=800, learning_rate=0.03, max_leaf_nodes=31,
                min_samples_leaf=20, l2_regularization=0.05, random_state=RANDOM_STATE)),
        "hgb_te_smooth": ModelConfig(
            name="hgb_te_smooth", encoding="target", dense=True, smoothing=80.0,
            estimator=HistGradientBoostingRegressor(
                max_iter=900, learning_rate=0.025, max_leaf_nodes=31,
                min_samples_leaf=30, l2_regularization=0.20, random_state=RANDOM_STATE)),
    }
    config = configs[model_name]
    preprocess = make_preprocessor(
        numeric_features=numeric_features, categorical_features=categorical_features,
        encoding=config.encoding, dense=config.dense, smoothing=config.smoothing,
        scale_numeric=config.scale_numeric)
    return Pipeline(steps=[("preprocess", preprocess), ("model", config.estimator)])


def get_cv(y, n_splits=5):
    y_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    return splitter, y_bins

print("Preprocessor ve CV yardımcıları tanımlandı.")


In [ ]:
# ─── Deney Kaydetme & Blend Fonksiyonları ─────────────────────────────────────

def _update_summary(row):
    root = project_root()
    summary_path = root / "reports" / "experiments" / "summary.csv"
    new_summary = pd.DataFrame([row])
    if summary_path.exists():
        summary = pd.read_csv(summary_path)
        summary = pd.concat([summary, new_summary], ignore_index=True)
        summary = summary.drop_duplicates(subset=["experiment_id"], keep="last")
    else:
        summary = new_summary
    summary.sort_values("cv_rmse").to_csv(summary_path, index=False)


def run_cv_experiment(experiment_id, feature_set, model_name, n_splits=5, save_submission=True):
    root = project_root()
    report_dir = root / "reports" / "experiments"
    submission_dir = root / "submissions"
    report_dir.mkdir(parents=True, exist_ok=True)
    submission_dir.mkdir(exist_ok=True)

    train, test = load_data()
    train_fe = apply_feature_set(train, feature_set)
    test_fe  = apply_feature_set(test,  feature_set)

    feature_cols = [col for col in train_fe.columns if col not in [TARGET, ID_COL]]
    X = train_fe[feature_cols]
    y = train_fe[TARGET]
    X_test = test_fe[feature_cols]

    numeric_features     = X.select_dtypes(include="number").columns.tolist()
    categorical_features = X.select_dtypes(exclude="number").columns.tolist()

    splitter, y_bins = get_cv(y, n_splits=n_splits)
    oof = np.zeros(len(X), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)
    fold_rows = []

    print(f"Experiment: {experiment_id}")
    print(f"  feature_set={feature_set}  model={model_name}")
    print(f"  features={len(feature_cols)}  numeric={len(numeric_features)}  categorical={len(categorical_features)}")

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(X, y_bins), start=1):
        model = clone(build_pipeline(model_name, numeric_features, categorical_features))
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]
        model.fit(X_tr, y_tr)
        valid_pred = np.clip(model.predict(X_val), 0, 10)
        fold_score = rmse(y_val, valid_pred)
        oof[valid_idx] = valid_pred
        test_pred += np.clip(model.predict(X_test), 0, 10) / n_splits
        fold_rows.append({"experiment_id": experiment_id, "feature_set": feature_set,
                          "model_name": model_name, "fold": fold, "rmse": fold_score})
        print(f"    fold {fold}: {fold_score:.6f}")

    cv_score = rmse(y, oof)
    print(f"  OOF RMSE: {cv_score:.6f}")

    oof_path        = report_dir / f"{experiment_id}_oof.csv"
    prediction_path = report_dir / f"{experiment_id}_test_predictions.csv"
    submission_path = submission_dir / f"{experiment_id}_{model_name}_{feature_set}.csv"

    pd.DataFrame(fold_rows).to_csv(report_dir / f"{experiment_id}_folds.csv", index=False)
    pd.DataFrame({ID_COL: train_fe[ID_COL], TARGET: y, "oof_pred": oof}).to_csv(oof_path, index=False)
    pd.DataFrame({ID_COL: test_fe[ID_COL], "prediction": np.clip(test_pred, 0, 10)}).to_csv(prediction_path, index=False)
    if save_submission:
        pd.DataFrame({ID_COL: test_fe[ID_COL], TARGET: np.clip(test_pred, 0, 10)}).to_csv(submission_path, index=False)

    _update_summary({
        "experiment_id": experiment_id, "feature_set": feature_set, "model_name": model_name,
        "n_splits": n_splits, "cv_rmse": cv_score,
        "fold_mean": float(np.mean([r["rmse"] for r in fold_rows])),
        "fold_std":  float(np.std( [r["rmse"] for r in fold_rows])),
        "oof_path": str(oof_path), "prediction_path": str(prediction_path),
        "submission_path": str(submission_path),
    })
    return {"experiment_id": experiment_id, "cv_rmse": cv_score,
            "oof_path": oof_path, "prediction_path": prediction_path}


def blend_experiments(experiment_id, source_experiment_ids, weights=None):
    root = project_root()
    report_dir   = root / "reports" / "experiments"
    submission_dir = root / "submissions"
    summary = pd.read_csv(report_dir / "summary.csv")

    if weights is None:
        weights = [1 / len(source_experiment_ids)] * len(source_experiment_ids)
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()

    oof_matrix, pred_matrix = [], []
    oof_base, pred_base = None, None
    for src_id in source_experiment_ids:
        row = summary.loc[summary["experiment_id"] == src_id].iloc[0]
        oof_df  = pd.read_csv(row["oof_path"])
        pred_df = pd.read_csv(row["prediction_path"])
        if oof_base is None:
            oof_base  = oof_df[[ID_COL, TARGET]].copy()
            pred_base = pred_df[[ID_COL]].copy()
        oof_matrix.append(oof_df["oof_pred"].to_numpy())
        pred_matrix.append(pred_df["prediction"].to_numpy())

    oof_pred  = np.clip(np.column_stack(oof_matrix)  @ weights, 0, 10)
    test_pred = np.clip(np.column_stack(pred_matrix) @ weights, 0, 10)
    cv_score  = rmse(oof_base[TARGET], oof_pred)

    oof_path        = report_dir / f"{experiment_id}_oof.csv"
    prediction_path = report_dir / f"{experiment_id}_test_predictions.csv"
    submission_path = submission_dir / f"{experiment_id}_blend.csv"

    oof_base["oof_pred"] = oof_pred
    pred_base["prediction"] = test_pred
    oof_base.to_csv(oof_path, index=False)
    pred_base.to_csv(prediction_path, index=False)
    pd.DataFrame({"source_experiment_id": source_experiment_ids, "weight": weights}).to_csv(
        report_dir / f"{experiment_id}_weights.csv", index=False)
    pd.DataFrame({ID_COL: pred_base[ID_COL], TARGET: test_pred}).to_csv(submission_path, index=False)

    _update_summary({
        "experiment_id": experiment_id, "feature_set": "blend", "model_name": "blend",
        "n_splits": float("nan"), "cv_rmse": cv_score,
        "fold_mean": float("nan"), "fold_std": float("nan"),
        "oof_path": str(oof_path), "prediction_path": str(prediction_path),
        "submission_path": str(submission_path),
    })
    print(f"Blend {experiment_id}  OOF RMSE: {cv_score:.6f}")
    return {"experiment_id": experiment_id, "cv_rmse": cv_score, "submission_path": submission_path}

print("run_cv_experiment ve blend_experiments tanımlandı.")


---
## 2. Veri Keşfi (EDA)

In [ ]:
train, test = load_data()
print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
train.head()


In [ ]:
# Hedef değişken istatistikleri
print(train[TARGET].describe())
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train[TARGET].hist(bins=40, ax=axes[0], color="#4C72B0", edgecolor="white")
axes[0].set_title("Hedef Dağılımı (train)")
axes[0].set_xlabel(TARGET)

train[TARGET].plot.box(ax=axes[1], color="#4C72B0")
axes[1].set_title("Kutu Grafiği")
plt.tight_layout()
plt.show()


In [ ]:
# Eksik değer özeti
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Eksik değer olan sütunlar:")
print(missing.to_string())


---
## 3. Stage 4 — HGB + Target Encoding Hiperparametre Taraması

`fs_07_all_safe` feature set üzerinde 9 farklı HGB konfigürasyonu çalıştırılır.  
Sonunda top-3 eşit blend + inverse-RMSE blend üretilir.


In [ ]:
# Stage 4 sabitleri
STAGE4_FEATURE_SET = "fs_07_all_safe"
STAGE4_MODEL_FAMILY = "hgb_te_tuned"
N_SPLITS = 5

HGB_TE_CONFIGS = [
    {"name":"base_recheck",       "smoothing":25.0,  "max_iter":800,  "learning_rate":0.030, "max_leaf_nodes":31, "min_samples_leaf":20, "l2_regularization":0.05},
    {"name":"smooth_regularized", "smoothing":80.0,  "max_iter":900,  "learning_rate":0.025, "max_leaf_nodes":31, "min_samples_leaf":30, "l2_regularization":0.20},
    {"name":"compact_fast",       "smoothing":25.0,  "max_iter":650,  "learning_rate":0.045, "max_leaf_nodes":15, "min_samples_leaf":20, "l2_regularization":0.03},
    {"name":"deeper_regularized", "smoothing":40.0,  "max_iter":900,  "learning_rate":0.025, "max_leaf_nodes":63, "min_samples_leaf":25, "l2_regularization":0.15},
    {"name":"low_lr_long",        "smoothing":25.0,  "max_iter":1100, "learning_rate":0.020, "max_leaf_nodes":31, "min_samples_leaf":20, "l2_regularization":0.05},
    {"name":"strong_l2",          "smoothing":80.0,  "max_iter":1000, "learning_rate":0.025, "max_leaf_nodes":31, "min_samples_leaf":40, "l2_regularization":0.40},
    {"name":"low_smoothing",      "smoothing":10.0,  "max_iter":800,  "learning_rate":0.030, "max_leaf_nodes":31, "min_samples_leaf":20, "l2_regularization":0.05},
    {"name":"high_smoothing",     "smoothing":150.0, "max_iter":800,  "learning_rate":0.030, "max_leaf_nodes":31, "min_samples_leaf":20, "l2_regularization":0.05},
    {"name":"smaller_leaf",       "smoothing":25.0,  "max_iter":800,  "learning_rate":0.030, "max_leaf_nodes":31, "min_samples_leaf":10, "l2_regularization":0.08},
]
print(f"Stage 4: {len(HGB_TE_CONFIGS)} config çalışacak.")


In [ ]:
def build_hgb_te_pipeline(config, numeric_features, categorical_features):
    preprocess = make_preprocessor(
        numeric_features=numeric_features, categorical_features=categorical_features,
        encoding="target", dense=True, smoothing=config["smoothing"])
    model = HistGradientBoostingRegressor(
        loss="squared_error",
        max_iter=config["max_iter"], learning_rate=config["learning_rate"],
        max_leaf_nodes=config["max_leaf_nodes"], min_samples_leaf=config["min_samples_leaf"],
        l2_regularization=config["l2_regularization"],
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=30,
        random_state=RANDOM_STATE)
    return Pipeline(steps=[("preprocess", preprocess), ("model", model)])


def run_hgb_config(config, train_fe, test_fe, X, y, X_test,
                   numeric_features, categorical_features, splitter, y_bins):
    root = project_root()
    report_dir   = root / "reports" / "experiments"
    submission_dir = root / "submissions"
    experiment_id = f"exp_4{config['idx']:02d}_hgb_te_{config['name']}"
    print(f"\n=== {experiment_id} ===")

    oof = np.zeros(len(X), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)
    fold_rows = []

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(X, y_bins), start=1):
        model = clone(build_hgb_te_pipeline(config, numeric_features, categorical_features))
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        valid_pred = np.clip(model.predict(X.iloc[valid_idx]), 0, 10)
        fold_score = rmse(y.iloc[valid_idx], valid_pred)
        oof[valid_idx] = valid_pred
        test_pred += np.clip(model.predict(X_test), 0, 10) / N_SPLITS
        fold_rows.append({"experiment_id": experiment_id, "feature_set": STAGE4_FEATURE_SET,
                          "model_name": STAGE4_MODEL_FAMILY, "fold": fold, "rmse": fold_score})
        print(f"  fold {fold}: {fold_score:.6f}")

    cv_score = rmse(y, oof)
    print(f"  OOF RMSE: {cv_score:.6f}")

    oof_path        = report_dir / f"{experiment_id}_oof.csv"
    prediction_path = report_dir / f"{experiment_id}_test_predictions.csv"
    submission_path = submission_dir / f"{experiment_id}_{STAGE4_MODEL_FAMILY}_{STAGE4_FEATURE_SET}.csv"

    pd.DataFrame(fold_rows).to_csv(report_dir / f"{experiment_id}_folds.csv", index=False)
    pd.DataFrame({ID_COL: train_fe[ID_COL], TARGET: y, "oof_pred": oof}).to_csv(oof_path, index=False)
    pd.DataFrame({ID_COL: test_fe[ID_COL], "prediction": np.clip(test_pred,0,10)}).to_csv(prediction_path, index=False)
    pd.DataFrame({ID_COL: test_fe[ID_COL], TARGET: np.clip(test_pred,0,10)}).to_csv(submission_path, index=False)
    pd.DataFrame([config]).to_csv(report_dir / f"{experiment_id}_params.csv", index=False)

    _update_summary({"experiment_id": experiment_id, "feature_set": STAGE4_FEATURE_SET,
        "model_name": STAGE4_MODEL_FAMILY, "n_splits": N_SPLITS, "cv_rmse": cv_score,
        "fold_mean": float(np.mean([r["rmse"] for r in fold_rows])),
        "fold_std":  float(np.std( [r["rmse"] for r in fold_rows])),
        "oof_path": str(oof_path), "prediction_path": str(prediction_path),
        "submission_path": str(submission_path)})

    return {"experiment_id": experiment_id, "config_name": config["name"], "cv_rmse": cv_score,
            "fold_mean": float(np.mean([r["rmse"] for r in fold_rows])),
            "fold_std":  float(np.std( [r["rmse"] for r in fold_rows]))}

print("HGB tuning fonksiyonu hazır.")


In [ ]:
# Stage 4 çalıştır
train, test = load_data()
train_fe = apply_feature_set(train, STAGE4_FEATURE_SET)
test_fe  = apply_feature_set(test,  STAGE4_FEATURE_SET)

feature_cols = [col for col in train_fe.columns if col not in [TARGET, ID_COL]]
X = train_fe[feature_cols]; y = train_fe[TARGET]; X_test = test_fe[feature_cols]
numeric_features     = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()
splitter, y_bins = get_cv(y, n_splits=N_SPLITS)

print("Stage 4 — HGB Target Encoding Tuning")
print(f"Feature set: {STAGE4_FEATURE_SET}")
print(f"features={len(feature_cols)}  numeric={len(numeric_features)}  categorical={len(categorical_features)}")

stage4_results = []
for idx, cfg in enumerate(HGB_TE_CONFIGS, start=1):
    c = dict(cfg); c["idx"] = idx
    stage4_results.append(run_hgb_config(c, train_fe, test_fe, X, y, X_test,
                                         numeric_features, categorical_features, splitter, y_bins))

result_df4 = pd.DataFrame(stage4_results).sort_values("cv_rmse").reset_index(drop=True)
print("\nStage 4 Sonuçları:")
print(result_df4[["experiment_id","config_name","cv_rmse","fold_mean","fold_std"]].to_string(index=False))


In [ ]:
# Stage 4 blend'leri
top3_ids = result_df4.head(3)["experiment_id"].tolist()
print("Top 3:", top3_ids)

blend_experiments("exp_490_hgb_te_tuned_top3_equal_blend", top3_ids)

inv = 1 / result_df4.head(3)["cv_rmse"].to_numpy()
inv_weights = (inv / inv.sum()).tolist()
blend_experiments("exp_491_hgb_te_tuned_top3_inverse_blend", top3_ids, weights=inv_weights)
print("Stage 4 blend'leri kaydedildi.")


---
## 4. Stage 6 — LightGBM / XGBoost / CatBoost Modelleri

`fs_07_all_safe` üzerinde 5 sklearn-pipeline modeli (LightGBM, XGBoost) ve 3 CatBoost native modeli çalıştırılır.


In [ ]:
STAGE6_FEATURE_SET = "fs_07_all_safe"

STAGE6_PIPELINE_CONFIGS = [
    {"experiment_id":"exp_601_lgbm_ohe_base", "model_name":"lgbm_ohe_base", "encoding":"onehot",
     "n_estimators":1800,"learning_rate":0.025,"num_leaves":31,"max_depth":-1,
     "min_child_samples":25,"subsample":0.85,"colsample_bytree":0.85,"reg_alpha":0.0,"reg_lambda":0.2},
    {"experiment_id":"exp_602_lgbm_te_base", "model_name":"lgbm_te_base", "encoding":"target", "smoothing":25.0,
     "n_estimators":1800,"learning_rate":0.025,"num_leaves":31,"max_depth":-1,
     "min_child_samples":25,"subsample":0.85,"colsample_bytree":0.85,"reg_alpha":0.0,"reg_lambda":0.2},
    {"experiment_id":"exp_603_lgbm_te_regularized", "model_name":"lgbm_te_regularized", "encoding":"target", "smoothing":80.0,
     "n_estimators":2200,"learning_rate":0.018,"num_leaves":24,"max_depth":-1,
     "min_child_samples":45,"subsample":0.80,"colsample_bytree":0.80,"reg_alpha":0.05,"reg_lambda":0.8},
    {"experiment_id":"exp_604_xgb_ohe_base", "model_name":"xgb_ohe_base", "encoding":"onehot",
     "n_estimators":1100,"learning_rate":0.025,"max_depth":4,"min_child_weight":8,
     "subsample":0.85,"colsample_bytree":0.85,"reg_alpha":0.0,"reg_lambda":1.5},
    {"experiment_id":"exp_605_xgb_te_base", "model_name":"xgb_te_base", "encoding":"target", "smoothing":25.0,
     "n_estimators":1100,"learning_rate":0.025,"max_depth":4,"min_child_weight":8,
     "subsample":0.85,"colsample_bytree":0.85,"reg_alpha":0.0,"reg_lambda":1.5},
]

STAGE6_CATBOOST_CONFIGS = [
    {"experiment_id":"exp_606_cat_native_base",    "model_name":"cat_native_base",
     "iterations":1600,"learning_rate":0.025,"depth":6,"l2_leaf_reg":4.0,"random_strength":1.0,"bagging_temperature":1.0},
    {"experiment_id":"exp_607_cat_native_depth5",  "model_name":"cat_native_depth5",
     "iterations":1800,"learning_rate":0.025,"depth":5,"l2_leaf_reg":6.0,"random_strength":1.0,"bagging_temperature":0.8},
    {"experiment_id":"exp_608_cat_native_depth7_reg","model_name":"cat_native_depth7_reg",
     "iterations":1400,"learning_rate":0.025,"depth":7,"l2_leaf_reg":8.0,"random_strength":1.5,"bagging_temperature":1.0},
]
print(f"Stage 6: {len(STAGE6_PIPELINE_CONFIGS)} pipeline + {len(STAGE6_CATBOOST_CONFIGS)} CatBoost config")


In [ ]:
def build_stage6_pipeline(config, numeric_features, categorical_features):
    model_name = config["model_name"]
    encoding   = config["encoding"]
    if model_name.startswith("lgbm"):
        from lightgbm import LGBMRegressor
        estimator = LGBMRegressor(
            objective="regression", n_estimators=config["n_estimators"],
            learning_rate=config["learning_rate"], num_leaves=config["num_leaves"],
            max_depth=config["max_depth"], min_child_samples=config["min_child_samples"],
            subsample=config["subsample"], colsample_bytree=config["colsample_bytree"],
            reg_alpha=config["reg_alpha"], reg_lambda=config["reg_lambda"],
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, force_col_wise=True)
    elif model_name.startswith("xgb"):
        from xgboost import XGBRegressor
        estimator = XGBRegressor(
            objective="reg:squarederror", eval_metric="rmse",
            n_estimators=config["n_estimators"], learning_rate=config["learning_rate"],
            max_depth=config["max_depth"], min_child_weight=config["min_child_weight"],
            subsample=config["subsample"], colsample_bytree=config["colsample_bytree"],
            reg_alpha=config["reg_alpha"], reg_lambda=config["reg_lambda"],
            random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist")
    else:
        raise ValueError(f"Desteklenmeyen model: {model_name}")

    preprocess = make_preprocessor(
        numeric_features=numeric_features, categorical_features=categorical_features,
        encoding=encoding, dense=False if encoding=="onehot" else True,
        smoothing=config.get("smoothing", 25.0))
    return Pipeline(steps=[("preprocess", preprocess), ("model", estimator)])


def prepare_catboost_frame(train_part, valid_part=None, test_part=None,
                            numeric_features=None, categorical_features=None):
    numeric_features     = numeric_features or []
    categorical_features = categorical_features or []
    train_out = train_part.copy()
    valid_out = valid_part.copy() if valid_part is not None else None
    test_out  = test_part.copy()  if test_part  is not None else None
    medians = train_out[numeric_features].median()
    train_out[numeric_features] = train_out[numeric_features].fillna(medians)
    if valid_out is not None: valid_out[numeric_features] = valid_out[numeric_features].fillna(medians)
    if test_out  is not None: test_out[numeric_features]  = test_out[numeric_features].fillna(medians)
    for col in categorical_features:
        for df in [x for x in [train_out, valid_out, test_out] if x is not None]:
            df[col] = df[col].astype("object").where(df[col].notna(), "Bilinmiyor").astype(str)
    return train_out, valid_out, test_out


def _save_outputs(experiment_id, model_name, feature_set, train_ids, test_ids, y, oof, test_pred, fold_rows, config):
    root = project_root()
    report_dir   = root / "reports" / "experiments"
    submission_dir = root / "submissions"
    cv_score  = rmse(y, oof)
    fold_scores = [r["rmse"] for r in fold_rows]
    oof_path        = report_dir / f"{experiment_id}_oof.csv"
    prediction_path = report_dir / f"{experiment_id}_test_predictions.csv"
    submission_path = submission_dir / f"{experiment_id}_{model_name}_{feature_set}.csv"
    pd.DataFrame(fold_rows).to_csv(report_dir / f"{experiment_id}_folds.csv", index=False)
    pd.DataFrame({ID_COL: train_ids, TARGET: y, "oof_pred": np.clip(oof,0,10)}).to_csv(oof_path, index=False)
    pd.DataFrame({ID_COL: test_ids, "prediction": np.clip(test_pred,0,10)}).to_csv(prediction_path, index=False)
    pd.DataFrame([config]).to_csv(report_dir / f"{experiment_id}_params.csv", index=False)
    pd.DataFrame({ID_COL: test_ids, TARGET: np.clip(test_pred,0,10)}).to_csv(submission_path, index=False)
    _update_summary({"experiment_id": experiment_id, "feature_set": feature_set,
        "model_name": model_name, "n_splits": N_SPLITS, "cv_rmse": cv_score,
        "fold_mean": float(np.mean(fold_scores)), "fold_std": float(np.std(fold_scores)),
        "oof_path": str(oof_path), "prediction_path": str(prediction_path),
        "submission_path": str(submission_path)})
    return {"experiment_id": experiment_id, "model_name": model_name, "cv_rmse": cv_score,
            "fold_mean": float(np.mean(fold_scores)), "fold_std": float(np.std(fold_scores)),
            "submission_path": str(submission_path)}

print("Stage 6 yardımcı fonksiyonları hazır.")


In [ ]:
def run_stage6_pipeline_config(config, train_fe, test_fe, X, y, X_test,
                              numeric_features, categorical_features, splitter, y_bins):
    experiment_id = config["experiment_id"]
    model_name    = config["model_name"]
    print(f"\n=== {experiment_id} ===")
    oof = np.zeros(len(X), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)
    fold_rows = []
    for fold, (train_idx, valid_idx) in enumerate(splitter.split(X, y_bins), start=1):
        model = clone(build_stage6_pipeline(config, numeric_features, categorical_features))
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        valid_pred = np.clip(model.predict(X.iloc[valid_idx]), 0, 10)
        fold_score = rmse(y.iloc[valid_idx], valid_pred)
        oof[valid_idx] = valid_pred
        test_pred += np.clip(model.predict(X_test), 0, 10) / N_SPLITS
        fold_rows.append({"experiment_id":experiment_id,"feature_set":STAGE6_FEATURE_SET,
                          "model_name":model_name,"fold":fold,"rmse":fold_score})
        print(f"  fold {fold}: {fold_score:.6f}")
    print(f"  OOF RMSE: {rmse(y, oof):.6f}")
    return _save_outputs(experiment_id, model_name, STAGE6_FEATURE_SET,
                         train_fe[ID_COL], test_fe[ID_COL], y, oof, test_pred, fold_rows, config)


def run_stage6_catboost_config(config, train_fe, test_fe, X, y, X_test,
                               numeric_features, categorical_features, splitter, y_bins):
    from catboost import CatBoostRegressor
    experiment_id = config["experiment_id"]
    model_name    = config["model_name"]
    print(f"\n=== {experiment_id} ===")

    feature_order = numeric_features + categorical_features
    cat_indices   = [feature_order.index(col) for col in categorical_features]
    X_ordered     = X[feature_order]
    X_test_ord    = X_test[feature_order]

    oof = np.zeros(len(X), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)
    fold_rows = []

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(X_ordered, y_bins), start=1):
        X_tr, X_val, X_te = prepare_catboost_frame(
            X_ordered.iloc[train_idx], X_ordered.iloc[valid_idx], X_test_ord,
            numeric_features=numeric_features, categorical_features=categorical_features)
        model = CatBoostRegressor(
            loss_function="RMSE", random_seed=RANDOM_STATE, allow_writing_files=False, verbose=False,
            **{k:v for k,v in config.items() if k not in ("experiment_id","model_name")})
        model.fit(X_tr, y.iloc[train_idx], cat_features=cat_indices)
        valid_pred = np.clip(model.predict(X_val), 0, 10)
        fold_score = rmse(y.iloc[valid_idx], valid_pred)
        oof[valid_idx] = valid_pred
        test_pred += np.clip(model.predict(X_te), 0, 10) / N_SPLITS
        fold_rows.append({"experiment_id":experiment_id,"feature_set":STAGE6_FEATURE_SET,
                          "model_name":model_name,"fold":fold,"rmse":fold_score})
        print(f"  fold {fold}: {fold_score:.6f}")
    print(f"  OOF RMSE: {rmse(y, oof):.6f}")
    return _save_outputs(experiment_id, model_name, STAGE6_FEATURE_SET,
                         train_fe[ID_COL], test_fe[ID_COL], y, oof, test_pred, fold_rows, config)

print("Stage 6 çalıştırma fonksiyonları hazır.")


In [ ]:
# Stage 6 çalıştır
train, test = load_data()
train_fe6 = apply_feature_set(train, STAGE6_FEATURE_SET)
test_fe6  = apply_feature_set(test,  STAGE6_FEATURE_SET)
feature_cols6 = [col for col in train_fe6.columns if col not in [TARGET, ID_COL]]
X6 = train_fe6[feature_cols6]; y6 = train_fe6[TARGET]; X_test6 = test_fe6[feature_cols6]
numeric_features6     = X6.select_dtypes(include="number").columns.tolist()
categorical_features6 = X6.select_dtypes(exclude="number").columns.tolist()
splitter6, y_bins6 = get_cv(y6, n_splits=N_SPLITS)

print("Stage 6 — External GBDT Models")
print(f"Feature set: {STAGE6_FEATURE_SET}")
print(f"features={len(feature_cols6)}  numeric={len(numeric_features6)}  categorical={len(categorical_features6)}")

stage6_results = []
for cfg in STAGE6_PIPELINE_CONFIGS:
    stage6_results.append(run_stage6_pipeline_config(
        cfg, train_fe6, test_fe6, X6, y6, X_test6,
        numeric_features6, categorical_features6, splitter6, y_bins6))

for cfg in STAGE6_CATBOOST_CONFIGS:
    stage6_results.append(run_stage6_catboost_config(
        cfg, train_fe6, test_fe6, X6, y6, X_test6,
        numeric_features6, categorical_features6, splitter6, y_bins6))

result_df6 = pd.DataFrame(stage6_results).sort_values("cv_rmse").reset_index(drop=True)
print("\nStage 6 Sonuçları:")
print(result_df6[["experiment_id","model_name","cv_rmse","fold_mean","fold_std"]].to_string(index=False))


In [ ]:
# Stage 6 blend'leri
top3_ext = result_df6.head(3)["experiment_id"].tolist()
print("Top 3 external:", top3_ext)
blend_experiments("exp_690_external_top3_equal_blend", top3_ext)

with_hgb = ["exp_491_hgb_te_tuned_top3_inverse_blend"] + top3_ext
blend_experiments("exp_691_hgb_external_top3_equal_blend", with_hgb)

summary6 = pd.read_csv(project_root() / "reports" / "experiments" / "summary.csv")
scores6 = [float(summary6.loc[summary6["experiment_id"]==eid,"cv_rmse"].iloc[0]) for eid in with_hgb]
inv6 = 1 / np.array(scores6)
blend_experiments("exp_692_hgb_external_top3_inverse_blend", with_hgb, weights=(inv6/inv6.sum()).tolist())
print("Stage 6 blend'leri kaydedildi.")


---
## 5. Stage 7 — CatBoost Feature Set Ablasyonu

Tüm feature set'ler üzerinde aynı CatBoost base modeli çalıştırılır.  
Top-3 feature set eşit blend olarak birleştirilir.


In [ ]:
CAT_BASE_CONFIG_S7 = {
    "iterations": 1600, "learning_rate": 0.025, "depth": 6,
    "l2_leaf_reg": 4.0, "random_strength": 1.0, "bagging_temperature": 1.0,
}


def run_catboost_feature_set(feature_set: str, idx: int):
    from catboost import CatBoostRegressor
    root = project_root()
    experiment_id = f"exp_7{idx:02d}_cat_native_base_{feature_set}"
    model_name    = "cat_native_base"

    train, test = load_data()
    train_fe = apply_feature_set(train, feature_set)
    test_fe  = apply_feature_set(test,  feature_set)
    feature_cols = [col for col in train_fe.columns if col not in [TARGET, ID_COL]]
    X = train_fe[feature_cols]; y = train_fe[TARGET]; X_test = test_fe[feature_cols]
    numeric_features     = X.select_dtypes(include="number").columns.tolist()
    categorical_features = X.select_dtypes(exclude="number").columns.tolist()
    feature_order = numeric_features + categorical_features
    cat_indices   = [feature_order.index(col) for col in categorical_features]

    splitter, y_bins = get_cv(y, n_splits=N_SPLITS)
    oof = np.zeros(len(X), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)
    fold_rows = []

    print(f"\n=== {experiment_id} ===")
    print(f"  feature_set={feature_set}  features={len(feature_cols)}")

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(X[feature_order], y_bins), start=1):
        X_tr, X_val, X_te = prepare_catboost_frame(
            X[feature_order].iloc[train_idx], X[feature_order].iloc[valid_idx], X_test[feature_order],
            numeric_features=numeric_features, categorical_features=categorical_features)
        model = CatBoostRegressor(loss_function="RMSE", random_seed=RANDOM_STATE,
                                   allow_writing_files=False, verbose=False, **CAT_BASE_CONFIG_S7)
        model.fit(X_tr, y.iloc[train_idx], cat_features=cat_indices)
        valid_pred = np.clip(model.predict(X_val), 0, 10)
        fold_score = rmse(y.iloc[valid_idx], valid_pred)
        oof[valid_idx] = valid_pred
        test_pred += np.clip(model.predict(X_te), 0, 10) / N_SPLITS
        fold_rows.append({"experiment_id":experiment_id,"feature_set":feature_set,
                          "model_name":model_name,"fold":fold,"rmse":fold_score})
        print(f"  fold {fold}: {fold_score:.6f}")

    cv_score = rmse(y, oof)
    print(f"  OOF RMSE: {cv_score:.6f}")
    return _save_outputs(experiment_id, model_name, feature_set,
                         train_fe[ID_COL], test_fe[ID_COL], y, oof, test_pred, fold_rows,
                         {**CAT_BASE_CONFIG_S7, "feature_set": feature_set})

print("Stage 7 fonksiyonu hazır.")


In [ ]:
# Stage 7 çalıştır
print("Stage 7 — CatBoost Feature Set Ablation")
feature_sets_all = available_feature_sets()
print("Feature sets:", feature_sets_all)

stage7_results = []
for idx, fs in enumerate(feature_sets_all, start=1):
    stage7_results.append(run_catboost_feature_set(feature_set=fs, idx=idx))

result_df7 = pd.DataFrame(stage7_results).sort_values("cv_rmse").reset_index(drop=True)
print("\nStage 7 Sonuçları:")
print(result_df7[["experiment_id","feature_set","cv_rmse"]].to_string(index=False))


In [ ]:
# Stage 7 blend (pipeline içindeki fixed blend — top3 sabit)
required_feature_runs = [
    ("fs_01_clean_country",      2),
    ("fs_02_missing_indicators", 3),
    ("fs_06_bins",               7),
]
for fs, orig_idx in required_feature_runs:
    run_catboost_feature_set(feature_set=fs, idx=orig_idx)

blend_experiments(
    "exp_790_cat_feature_top3_equal_blend",
    source_experiment_ids=[
        "exp_703_cat_native_base_fs_02_missing_indicators",
        "exp_707_cat_native_base_fs_06_bins",
        "exp_702_cat_native_base_fs_01_clean_country",
    ])
print("Stage 7 blend kaydedildi.")


---
## 6. Stage 8 — CatBoost Fine-Tuning + Seed Averaging

8 farklı CatBoost konfigürasyonu `fs_02_missing_indicators` üzerinde çalıştırılır.  
En iyi config 5 farklı seed ile tekrar çalıştırılır ve ortalaması alınır.


In [ ]:
import os
STAGE8_FEATURE_SET = "fs_02_missing_indicators"
STAGE8_MODEL_NAME  = "cat_native_tuned"
SEEDS = [42, 11, 77, 2026, 3407]
CATBOOST_TASK_TYPE = os.getenv("CATBOOST_TASK_TYPE", "CPU").upper()
CATBOOST_DEVICES   = os.getenv("CATBOOST_DEVICES", "0")
FAST_MODE = os.getenv("FAST_MODE", "0") == "1"

CAT_TUNING_CONFIGS = [
    {"name":"base_es",             "iterations":2500,"learning_rate":0.025,"depth":6,"l2_leaf_reg":4.0,"random_strength":1.0,"bagging_temperature":1.0},
    {"name":"depth5_reg",          "iterations":3000,"learning_rate":0.020,"depth":5,"l2_leaf_reg":6.0,"random_strength":1.0,"bagging_temperature":0.8},
    {"name":"depth6_reg",          "iterations":2800,"learning_rate":0.022,"depth":6,"l2_leaf_reg":8.0,"random_strength":1.2,"bagging_temperature":0.8},
    {"name":"depth7_reg",          "iterations":2400,"learning_rate":0.020,"depth":7,"l2_leaf_reg":8.0,"random_strength":1.5,"bagging_temperature":1.0},
    {"name":"depth4_long",         "iterations":3500,"learning_rate":0.018,"depth":4,"l2_leaf_reg":5.0,"random_strength":1.0,"bagging_temperature":0.8},
    {"name":"faster_depth6",       "iterations":1800,"learning_rate":0.035,"depth":6,"l2_leaf_reg":4.0,"random_strength":1.0,"bagging_temperature":1.0},
    {"name":"low_random_strength", "iterations":2500,"learning_rate":0.025,"depth":6,"l2_leaf_reg":4.0,"random_strength":0.3,"bagging_temperature":1.0},
    {"name":"high_random_strength","iterations":2500,"learning_rate":0.025,"depth":6,"l2_leaf_reg":4.0,"random_strength":2.0,"bagging_temperature":1.2},
]

def catboost_runtime_params():
    if CATBOOST_TASK_TYPE == "GPU":
        return {"task_type":"GPU","devices":CATBOOST_DEVICES}
    return {}

print(f"Stage 8: {len(CAT_TUNING_CONFIGS)} config, {len(SEEDS)} seed")
print(f"FAST_MODE={FAST_MODE}, task_type={CATBOOST_TASK_TYPE}")


In [ ]:
def run_catboost_cv_s8(experiment_id, feature_set, config, random_seed):
    from catboost import CatBoostRegressor
    train, test = load_data()
    train_fe = apply_feature_set(train, feature_set)
    test_fe  = apply_feature_set(test,  feature_set)
    feature_cols = [col for col in train_fe.columns if col not in [TARGET, ID_COL]]
    X = train_fe[feature_cols]; y = train_fe[TARGET]; X_test = test_fe[feature_cols]
    numeric_features     = X.select_dtypes(include="number").columns.tolist()
    categorical_features = X.select_dtypes(exclude="number").columns.tolist()
    feature_order = numeric_features + categorical_features
    cat_indices   = [feature_order.index(col) for col in categorical_features]
    X_ord = X[feature_order]; X_test_ord = X_test[feature_order]
    splitter, y_bins = get_cv(y, n_splits=N_SPLITS)
    oof = np.zeros(len(X), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)
    fold_rows = []
    print(f"\n=== {experiment_id} === seed={random_seed}")

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(X_ord, y_bins), start=1):
        X_tr, X_val, X_te = prepare_catboost_frame(
            X_ord.iloc[train_idx], X_ord.iloc[valid_idx], X_test_ord,
            numeric_features=numeric_features, categorical_features=categorical_features)
        model = CatBoostRegressor(
            loss_function="RMSE", random_seed=random_seed, allow_writing_files=False,
            verbose=False, od_type="Iter", od_wait=120, use_best_model=True,
            **catboost_runtime_params(),
            **{k:v for k,v in config.items() if k != "name"})
        model.fit(X_tr, y.iloc[train_idx], cat_features=cat_indices,
                  eval_set=(X_val, y.iloc[valid_idx]))
        valid_pred = np.clip(model.predict(X_val), 0, 10)
        fold_score = rmse(y.iloc[valid_idx], valid_pred)
        oof[valid_idx] = valid_pred
        test_pred += np.clip(model.predict(X_te), 0, 10) / N_SPLITS
        fold_rows.append({"experiment_id":experiment_id,"feature_set":feature_set,
                          "model_name":STAGE8_MODEL_NAME,"config_name":config["name"],
                          "seed":random_seed,"fold":fold,"rmse":fold_score})
        print(f"  fold {fold}: {fold_score:.6f}")

    print(f"  OOF RMSE: {rmse(y, oof):.6f}")
    return _save_outputs(experiment_id, STAGE8_MODEL_NAME, feature_set,
                         train_fe[ID_COL], test_fe[ID_COL], y, oof, test_pred, fold_rows,
                         {**config, "feature_set": feature_set, "seed": random_seed})

print("Stage 8 fonksiyonu hazır.")


In [ ]:
# Stage 8 çalıştır
print("Stage 8 — CatBoost Tuning + Seed Averaging")
configs_to_run = CAT_TUNING_CONFIGS[:3] if FAST_MODE else CAT_TUNING_CONFIGS
seeds_to_run   = SEEDS[:2] if FAST_MODE else SEEDS
print(f"config sayısı={len(configs_to_run)}, seed sayısı={len(seeds_to_run)}")

tuning_results = []
for idx, cfg in enumerate(configs_to_run, start=1):
    experiment_id = f"exp_8{idx:02d}_cat_tuned_{cfg['name']}"
    tuning_results.append(run_catboost_cv_s8(experiment_id, STAGE8_FEATURE_SET, cfg, RANDOM_STATE))

result_df8 = pd.DataFrame(tuning_results).sort_values("cv_rmse").reset_index(drop=True)
print("\nStage 8 Tuning Sonuçları:")
print(result_df8[["experiment_id","config_name","cv_rmse","fold_mean","fold_std"]].to_string(index=False))


In [ ]:
# En iyi config ile seed averaging
best_cfg_name = result_df8.iloc[0]["config_name"]
best_cfg = next(c for c in configs_to_run if c["name"] == best_cfg_name)
print(f"Seed averaging için en iyi config: {best_cfg_name}")

seed_experiment_ids = []
for seed in seeds_to_run:
    eid = f"exp_850_cat_seed_{seed}_{best_cfg_name}"
    run_catboost_cv_s8(eid, STAGE8_FEATURE_SET, best_cfg, seed)
    seed_experiment_ids.append(eid)

blend_experiments("exp_890_cat_seed_average_blend", seed_experiment_ids)
blend_experiments("exp_891_hgb_cat_seed_average_blend",
                  ["exp_491_hgb_te_tuned_top3_inverse_blend", "exp_890_cat_seed_average_blend"])
blend_experiments("exp_892_hgb_cat_seed_feature_blend",
                  ["exp_491_hgb_te_tuned_top3_inverse_blend",
                   "exp_890_cat_seed_average_blend",
                   "exp_790_cat_feature_top3_equal_blend"])
print("Stage 8 blend'leri kaydedildi.")


---
## 7. Stage 9 — Seçilmiş Blend Optimizasyonu

OOF tahminleri üzerinde:
1. Core-3 grid search (0.0025 adım)
2. Dirichlet random search (4 alpha değeri)
3. Pairwise refinement

Üç farklı model havuzu için uygulanır.


In [ ]:
STAGE9_RANDOM_ITER = int(os.getenv("STAGE9_RANDOM_ITER", "120000"))

CORE3_IDS = [
    "exp_491_hgb_te_tuned_top3_inverse_blend",
    "exp_890_cat_seed_average_blend",
    "exp_790_cat_feature_top3_equal_blend",
]
SELECTED_IDS = [
    "exp_491_hgb_te_tuned_top3_inverse_blend",
    "exp_890_cat_seed_average_blend",
    "exp_790_cat_feature_top3_equal_blend",
    "exp_802_cat_tuned_depth5_reg",
    "exp_808_cat_tuned_high_random_strength",
    "exp_605_xgb_te_base",
    "exp_603_lgbm_te_regularized",
]
DIVERSE_IDS = [
    "exp_491_hgb_te_tuned_top3_inverse_blend",
    "exp_890_cat_seed_average_blend",
    "exp_790_cat_feature_top3_equal_blend",
    "exp_802_cat_tuned_depth5_reg",
    "exp_808_cat_tuned_high_random_strength",
    "exp_605_xgb_te_base",
    "exp_604_xgb_ohe_base",
    "exp_603_lgbm_te_regularized",
    "exp_606_cat_native_base",
]
print(f"Stage 9: RANDOM_ITER={STAGE9_RANDOM_ITER}")


In [ ]:
def score_weights(y, oof_matrix, weights):
    pred = np.clip(oof_matrix @ weights, 0, 10)
    return rmse(y, pred)


def load_summary():
    summary_path = project_root() / "reports" / "experiments" / "summary.csv"
    summary = pd.read_csv(summary_path)
    summary["cv_rmse"] = summary["cv_rmse"].astype(float)
    return summary


def filter_existing_ids(summary, source_ids):
    available = set(summary["experiment_id"])
    existing = [eid for eid in source_ids if eid in available]
    missing  = [eid for eid in source_ids if eid not in available]
    if missing:
        print("  Atlanıyor (mevcut değil):", missing)
    return existing


def load_predictions(summary, source_ids):
    oof_matrix, pred_matrix = [], []
    oof_base = pred_base = None
    for eid in source_ids:
        row = summary.loc[summary["experiment_id"] == eid].iloc[0]
        oof_df  = pd.read_csv(row["oof_path"])
        pred_df = pd.read_csv(row["prediction_path"])
        if oof_base is None:
            oof_base  = oof_df[[ID_COL, TARGET]].copy()
            pred_base = pred_df[[ID_COL]].copy()
        oof_matrix.append(oof_df["oof_pred"].to_numpy())
        pred_matrix.append(pred_df["prediction"].to_numpy())
    return oof_base, pred_base, np.column_stack(oof_matrix), np.column_stack(pred_matrix)


def save_blend_s9(experiment_id, source_ids, weights, oof_base, pred_base, oof_matrix, pred_matrix):
    root = project_root()
    report_dir   = root / "reports" / "experiments"
    submission_dir = root / "submissions"
    weights = np.array(weights, dtype=float) / np.array(weights).sum()
    oof_pred  = np.clip(oof_matrix  @ weights, 0, 10)
    test_pred = np.clip(pred_matrix @ weights, 0, 10)
    score = rmse(oof_base[TARGET], oof_pred)

    oof_path        = report_dir / f"{experiment_id}_oof.csv"
    prediction_path = report_dir / f"{experiment_id}_test_predictions.csv"
    submission_path = submission_dir / f"{experiment_id}_blend.csv"
    oof = oof_base.copy(); oof["oof_pred"] = oof_pred
    pred = pred_base.copy(); pred["prediction"] = test_pred
    oof.to_csv(oof_path, index=False)
    pred.to_csv(prediction_path, index=False)
    pd.DataFrame({"source_experiment_id": source_ids, "weight": weights}).to_csv(
        report_dir / f"{experiment_id}_weights.csv", index=False)
    pd.DataFrame({ID_COL: pred[ID_COL], TARGET: test_pred}).to_csv(submission_path, index=False)

    _update_summary({"experiment_id":experiment_id,"feature_set":"blend","model_name":"blend",
        "n_splits":float("nan"),"cv_rmse":score,"fold_mean":float("nan"),"fold_std":float("nan"),
        "oof_path":str(oof_path),"prediction_path":str(prediction_path),"submission_path":str(submission_path)})
    return {"experiment_id":experiment_id,"cv_rmse":score,"submission_path":str(submission_path)}


def core3_grid_search(y, oof_matrix, step=0.0025):
    best_score, best_weights = float("inf"), None
    n_steps = int(round(1 / step))
    for i in range(n_steps + 1):
        w1 = i * step
        for j in range(int(round((1-w1)/step)) + 1):
            w2 = j * step; w3 = 1 - w1 - w2
            if w3 < -1e-12: continue
            weights = np.array([w1, w2, w3], dtype=float)
            s = score_weights(y, oof_matrix, weights)
            if s < best_score: best_score, best_weights = s, weights
    return best_score, best_weights


def random_dirichlet_search(y, oof_matrix, n_iter, alpha, max_weight=None):
    rng = np.random.default_rng(RANDOM_STATE + int(alpha * 1000))
    n_models = oof_matrix.shape[1]
    best_score, best_weights = float("inf"), None
    for _ in range(n_iter):
        weights = rng.dirichlet(np.ones(n_models) * alpha)
        if max_weight is not None and weights.max() > max_weight: continue
        s = score_weights(y, oof_matrix, weights)
        if s < best_score: best_score, best_weights = s, weights
    return best_score, best_weights


def pairwise_refine(y, oof_matrix, initial_weights, step_sizes=(0.02, 0.01, 0.005, 0.002, 0.001)):
    weights = initial_weights.copy().astype(float) / initial_weights.sum()
    best_score = score_weights(y, oof_matrix, weights)
    for step in step_sizes:
        improved = True
        while improved:
            improved = False; best_candidate = None
            for si in range(len(weights)):
                if weights[si] < step: continue
                for ti in range(len(weights)):
                    if si == ti: continue
                    c = weights.copy(); c[si] -= step; c[ti] += step
                    s = score_weights(y, oof_matrix, c)
                    if s < best_score - 1e-10: best_candidate = c; best_score = s
            if best_candidate is not None: weights = best_candidate; improved = True
    return best_score, weights


def run_pool(pool_name, experiment_id_prefix, source_ids, summary):
    source_ids = filter_existing_ids(summary, source_ids)
    oof_base, pred_base, oof_matrix, pred_matrix = load_predictions(summary, source_ids)
    y = oof_base[TARGET].to_numpy()
    print(f"\n=== {pool_name} ===")

    results = []
    equal_weights = np.ones(len(source_ids)) / len(source_ids)
    results.append(save_blend_s9(f"{experiment_id_prefix}_equal_blend",
                                  source_ids, equal_weights, oof_base, pred_base, oof_matrix, pred_matrix))

    best_r_score, best_r_weights = float("inf"), None
    for alpha in [0.4, 0.8, 1.5, 3.0]:
        s, w = random_dirichlet_search(y, oof_matrix, STAGE9_RANDOM_ITER, alpha, max_weight=0.70)
        print(f"  random alpha={alpha}: {s:.6f}")
        if s < best_r_score: best_r_score, best_r_weights = s, w

    results.append(save_blend_s9(f"{experiment_id_prefix}_random_blend",
                                  source_ids, best_r_weights, oof_base, pred_base, oof_matrix, pred_matrix))

    refined_score, refined_weights = pairwise_refine(y, oof_matrix, best_r_weights)
    print(f"  pairwise refined: {refined_score:.6f}")
    results.append(save_blend_s9(f"{experiment_id_prefix}_refined_blend",
                                  source_ids, refined_weights, oof_base, pred_base, oof_matrix, pred_matrix))
    return results

print("Stage 9 fonksiyonları hazır.")


In [ ]:
# Stage 9 çalıştır
print("Stage 9 — Selected Blend Optimization")
print(f"RANDOM_ITER: {STAGE9_RANDOM_ITER}")
summary9 = load_summary()
all_s9_results = []

# Core-3 grid search
core_ids = filter_existing_ids(summary9, CORE3_IDS)
oof_base9, pred_base9, oof_matrix9, pred_matrix9 = load_predictions(summary9, core_ids)
y9 = oof_base9[TARGET].to_numpy()
print("\n=== core3_grid ===")
grid_score, grid_weights = core3_grid_search(y9, oof_matrix9)
print(f"core3 grid: {grid_score:.6f}")
all_s9_results.append(save_blend_s9("exp_900_core3_grid_blend", core_ids, grid_weights,
                                     oof_base9, pred_base9, oof_matrix9, pred_matrix9))

# Havuzlar
summary9 = load_summary()
all_s9_results.extend(run_pool("core3_random",    "exp_901_core3",    CORE3_IDS,    summary9))
summary9 = load_summary()
all_s9_results.extend(run_pool("selected_random", "exp_902_selected", SELECTED_IDS, summary9))
summary9 = load_summary()
all_s9_results.extend(run_pool("diverse_random",  "exp_903_diverse",  DIVERSE_IDS,  summary9))

result_df9 = pd.DataFrame(all_s9_results).sort_values("cv_rmse").reset_index(drop=True)
print("\nStage 9 Sonuçları:")
print(result_df9[["experiment_id","cv_rmse"]].to_string(index=False))


---
## 8. Final Submission

En iyi submission dosyasını `final_submission.csv` olarak kopyala.


In [ ]:
FINAL_SUBMISSION_NAME = "exp_903_diverse_refined_blend_blend.csv"

src = project_root() / "submissions" / FINAL_SUBMISSION_NAME
dst = project_root() / "submissions" / "final_submission.csv"

if src.exists():
    shutil.copy(src, dst)
    print(f"Final submission kaydedildi: {dst}")
    final_df = pd.read_csv(dst)
    print(f"Satır sayısı: {len(final_df)}")
    print(final_df.head())
else:
    print(f"UYARI: {src} bulunamadı. Pipeline henüz tamamlanmadı.")


---
## 9. Sonuç Özeti

In [ ]:
# Tüm deneylerin özet tablosu
summary_path = project_root() / "reports" / "experiments" / "summary.csv"
if summary_path.exists():
    summary_final = pd.read_csv(summary_path).sort_values("cv_rmse").reset_index(drop=True)
    print(f"Toplam deney sayısı: {len(summary_final)}")
    print("\nEn iyi 15 deney:")
    print(summary_final[["experiment_id","feature_set","model_name","cv_rmse","fold_mean","fold_std"]].head(15).to_string(index=False))
else:
    print("Özet dosyası henüz oluşturulmadı.")


In [ ]:
# Final blend ağırlık tablosu
weights_path = project_root() / "reports" / "experiments" / "exp_903_diverse_refined_blend_weights.csv"
if weights_path.exists():
    weights_df = pd.read_csv(weights_path).sort_values("weight", ascending=False).reset_index(drop=True)
    print("Final Blend Ağırlıkları (exp_903_diverse_refined_blend):")
    print(weights_df.to_string(index=False))
else:
    print("Ağırlık dosyası henüz oluşturulmadı.")


---
## Notlar

- **Validation:** Stratified 5-fold CV (hedef değişkenin quantile bin'leriyle dengeli)
- **Final Local CV:** `1.21350`
- **Final Public RMSE:** `1.20244`
- **En güçlü model ailesi:** CatBoost (native categorical desteği sayesinde)
- **Kazanım kaynağı:** Tek büyük model değil, güvenilir OOF blend optimizasyonu
